# 03 · FunnyBirds + MCBM — does minimality fix grounding?

**Claim under test (MCBM).** The information bottleneck makes each `z_j` a
*minimal sufficient statistic* of `c_j` — "there was never a bottleneck in CBMs,"
minimality is what makes concepts faithful. **Our question:** minimality
constrains z's *content* (`I(z_j;x|c_j)=0`); backwash is a *source* problem
(which pixels z reads). When `c=f(class)`, encoding only `c_j` is satisfiable by
class-lookup — so minimality need not remove backwash.

Same backbone & data as CBM; only the head + γ change. Effective IB force = **γ×0.2**.
**γ=0 is MCBM-with-no-IB, not vanilla CBM** (different head). γ values are the
paper's own (CUB 0.05–0.3, synthetic 1–5).

Reference: `fb_mcbm_renderer_swap.ipynb`, `fb_mcbm_rl_renderer_swap.ipynb`
(§19 fwd/bwd heatmap, §20 IB-compression-vs-grounding).

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CURATED = Path(os.environ["CURATED_DATA"])            # cluster data root
REPO = Path.cwd().parent                              # run from curated/notebooks
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"

def need(p, how):
    p = Path(p)
    if not p.exists():
        print(f"[pending] {p}\n  produce it:  {how}")
    return p.exists()


## 1 · Backwash vs γ — the money figure
`analysis/collect_backwash.py` → `backwash_vs_gamma.csv`. Prediction: backwash
**persists** across γ (minimality does not fix the source problem); the CBM
reference line is drawn for scale.

In [ ]:
bw = CURATED/"backwash_vs_gamma.csv"
if need(bw, 'GAMMAS="0 0.1 0.3 1 3 5" bash train/run_all_funnybirds.sh   (trains sweep + collects)'):
    T = pd.read_csv(bw); display(T.round(3))
    mc = T[T.model=="mcbm"].copy(); cb = T[T.model=="cbm"]
    g = mc.groupby("gamma").backwash.agg(["mean","std"]).reset_index()
    floor = (g.gamma[g.gamma>0].min() or 0.05)/3
    x = g.gamma.replace(0, floor)
    fig,ax=plt.subplots(figsize=(6.5,4))
    ax.errorbar(x, g["mean"], yerr=g["std"].fillna(0), marker="o", color=MCBM_C, label="MCBM")
    if len(cb): ax.axhline(cb.backwash.mean(), ls="--", color=CBM_C, label="CBM (ref)")
    ax.set_xscale("log"); ax.set_xlabel("γ  (IB strength; effective force γ×0.2)")
    ax.set_ylabel("backwash  (retained P of removed part)"); ax.set_ylim(0,1)
    ax.set_title("Concept–class backwash vs bottleneck strength\n(FunnyBirds · deletion test)"); ax.legend()

## 2 · Species-code vs γ — does the bottleneck stop being a class code?
`species_probe/funnybirds-mcbm-g*-s*.json`. If `species←c_preds` stays high as γ
grows, minimality compressed the representation without cutting the class channel.

In [ ]:
import re
SEED=1; rows=[]
for f in sorted((CURATED/"species_probe").glob("funnybirds-mcbm-g*-s%d.json"%SEED)):
    m=re.search(r"-g([0-9p]+)-s",f.name);
    if not m: continue
    S=json.loads(f.read_text())
    rows.append(dict(gamma=float(m.group(1).replace("p",".")),
                     z=S["species_from_z"]["acc"], c=S["species_from_cpreds"]["acc"], chance=S["chance"]))
if rows:
    D=pd.DataFrame(rows).sort_values("gamma"); display(D.round(3))
    fig,ax=plt.subplots(figsize=(6,3.6))
    ax.plot(D.gamma.replace(0,0.02), D.c, "o-", color=MCBM_C, label="species←c_preds")
    ax.plot(D.gamma.replace(0,0.02), D.z, "s--", color="#888", label="species←z")
    ax.axhline(D.chance.iloc[0], ls=":", color="k", label="chance"); ax.set_xscale("log")
    ax.set_xlabel("γ"); ax.set_ylabel("species recoverable"); ax.set_ylim(0,1); ax.legend()
else:
    print("[pending] no MCBM species_probe json yet -> run the sweep (grounding_sweep.sh runs the probe too).")

**Takeaway.** If backwash is flat/rising in γ while `species←c_preds` stays high,
minimality shaped *content* but left the *source* (class channel) intact — the
core refutation: a minimal sufficient statistic of a class-derived label is still
a class code. Compare against CBM (notebook 02).